# IMPORT STATEMENTS
Below are the different imports for the demo, both from classes created for the project and for external libraries.

In [1]:
from classes.neural_network import NeuralNetwork, FeedForwardLayer
from classes.trainer import Trainer
from classes.recommender import Recommender
from classes.pytorch_trainer import PyTorchTrainer
from classes.autoencoder import Autoencoder

import numpy as np
import matplotlib.pyplot as plt
import torch

import os

/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# PIPELINE 1
### NEURAL NETWORK CREATION DEMO
Below is a demo for how to create a neural network using the NeuralNetwork class of this project. We are creating a deep autoencoder here with 4 encoder layers and 4 decoder layers. The embedding generated by the encoder section in this network is 32 dimensional. Further down in this notebook, we demonstrate how to train this model.

In [2]:
bias_scale = 0.0001
nn = NeuralNetwork(
    input_size=202,
    output_size=202
)
# Layer 1
layer_1_weights = Trainer.get_he_initialization(202, 165)
layer_1_biases = np.random.rand(165) * bias_scale
layer_1 = FeedForwardLayer(layer_1_weights, layer_1_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 2
layer_2_weights = Trainer.get_he_initialization(165, 128)
layer_2_biases = np.random.rand(128) * bias_scale
layer_2 = FeedForwardLayer(layer_2_weights, layer_2_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 3
layer_3_weights = Trainer.get_he_initialization(128, 64)
layer_3_biases = np.random.rand(64) * bias_scale
layer_3 = FeedForwardLayer(layer_3_weights, layer_3_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 4
layer_4_weights = Trainer.get_he_initialization(64, 32)
layer_4_biases = np.random.rand(32) * bias_scale
layer_4 = FeedForwardLayer(layer_4_weights, layer_4_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 5
layer_5_weights = Trainer.get_he_initialization(32, 64)
layer_5_biases = np.random.rand(64) * bias_scale
layer_5 = FeedForwardLayer(layer_5_weights, layer_5_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 6
layer_6_weights = Trainer.get_he_initialization(64, 128)
layer_6_biases = np.random.rand(128) * bias_scale
layer_6 = FeedForwardLayer(layer_6_weights, layer_6_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 7
layer_7_weights = Trainer.get_he_initialization(128, 165)
layer_7_biases = np.random.rand(165) * bias_scale
layer_7 = FeedForwardLayer(layer_7_weights, layer_7_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Layer 8
layer_8_weights = Trainer.get_he_initialization(165, 202)
layer_8_biases = np.random.rand(202) * bias_scale
layer_8 = FeedForwardLayer(layer_8_weights, layer_8_biases, NeuralNetwork.relu, NeuralNetwork.relu_dx)
# Add each layer
nn.append_layer(layer_1)
nn.append_layer(layer_2)
nn.append_layer(layer_3)
nn.append_layer(layer_4)
nn.append_layer(layer_5)
nn.append_layer(layer_6)
nn.append_layer(layer_7)
nn.append_layer(layer_8)

### CREATION OF TRAINER OBJECT
Below an instance of the Trainer class is created. This class allows for training a model; retrieving data from the dataset, and preprocessing it. This object is used below in many cells.

In [3]:
# Create trainer object
trainer = Trainer(
    training_network=nn,
    initial_lr=0.001,
    final_lr=0.0001,
    momentum=0.9,
    l2_regularization_lambda=0,
    num_epochs=2,
    dataset_path='ProjectDataset',
    output_folder='./networks/demo_networks'
)

### DATASET PREPROCESSING
The dataset is a primary dataset constructed by combining the Spotify Tracks Dataset (https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset) and the Million Song Dataset. Code showing how this is done is in classes/utils/merge_msd.py. Anyway, here is a demo of 5 tracks where we check if they are missing values. The values in the dataset are already normalized.

In [4]:
# get first 5 file paths
print("Displaying first 5 songs in dataset...")
file_paths = trainer.get_file_paths(input_dim=202, num_files=5)
for file_path in file_paths:
    print()
    song = trainer.get_song_data_from_file(file_path)
    if trainer.is_missing_values(song):
        print(f"Song {song.song_name} is missing values")
    print(song.to_string())

Displaying first 5 songs in dataset...

"Electro Glide In Blue" by Apollo 440
Key/Mode: 9/0
155.77 BPM in 4 time
Danceability: 0.521
Energy: 0.899
Loudness: -4.408
Valence: 0.653
Instrumentalness: 0.112

"Crashing Foreign Cars" by Helmet
Key/Mode: 7/1
165.226 BPM in 3 time
Danceability: 0.533
Energy: 0.965
Loudness: -3.371
Valence: 0.34
Instrumentalness: 0.0

"Here Without You" by 3 Doors Down
Key/Mode: 10/0
143.697 BPM in 4 time
Danceability: 0.556
Energy: 0.545
Loudness: -6.768
Valence: 0.193
Instrumentalness: 0.0

"Can't Satisfy Her" by I Wayne
Key/Mode: 7/1
116.303 BPM in 5 time
Danceability: 0.719
Energy: 0.656
Loudness: -6.155
Valence: 0.774
Instrumentalness: 0.0

"Fade to Grey" by Visage
Key/Mode: 4/0
125.127 BPM in 3 time
Danceability: 0.604
Energy: 0.416
Loudness: -17.154
Valence: 0.512
Instrumentalness: 0.0165


### TRAINING THE MODEL
Below is a demo of the model created previously ("nn") being trained for 2 epochs. During training, checkpoint saves occur every 10 epochs (which you will not see here), and if a model's validation loss and/or training loss is the best seen so far, the model is also saved. Also, the model is saved after the final epoch is run. The training and validation losses per epoch are both plotted, and the training and validation times elapsed per epoch are also plotted. These plots can be saved due to the functionality that matplotlib, the library used for plotting, provides. The raw scores are also saved in CSV files.

In [5]:
# train model
trainer.train_model()


Fetching training and validation data...
Number of training items: 4137
Number of validation items: 1034

Starting training...

Epoch 1...


Training cycle progress:  39%|█████████████████████████████████                                                   | 1631/4137 [00:36<00:56, 44.10it/s]


KeyboardInterrupt: 

### LOADING A NEURAL NETWORK FROM A FILE
Here is how a neural network is loaded. The weights, bias vectors, and activation functions for every layer are stored in a Pickle file, and the network is reconstructed using this file.

In [7]:
encoder: NeuralNetwork = NeuralNetwork.load_network('/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/networks/encoder_experiment_1/encoder.pkl')
print(encoder)

..................................................................
LAYER 1:
Weights:
[[ 0.07584737  0.2906666   0.20572838 ... -0.00968949 -0.00966685
   0.04598059]
 [ 0.12708169 -0.03171394 -0.01486029 ... -0.13104592  0.18163958
  -0.0332947 ]
 [-0.05181369  0.06539536 -0.03824587 ...  0.06171291 -0.18561054
  -0.03758846]
 ...
 [-0.02062382  0.05006193  0.22458385 ... -0.06297471  0.04289114
  -0.02933438]
 [ 0.22341425 -0.11636285 -0.00326176 ...  0.12847744 -0.06949709
   0.18927252]
 [-0.08577224  0.09301142 -0.08125531 ...  0.16578856 -0.08959377
   0.09482795]]
Bias:
[ 1.58845113e-01  1.23317858e-01 -1.58782452e-02  1.22563950e-01
  4.93828358e-02 -1.33454059e-02 -2.21768843e-02  1.56045573e-02
 -2.68095444e-02 -1.34130064e-03  1.08699278e-01  1.44745545e-02
  4.33852605e-02  3.63412106e-02  7.57360308e-02 -6.89792658e-03
  7.65435373e-02 -4.03510089e-03  4.74158541e-02 -4.98000727e-03
 -7.71100638e-03  1.00134197e-01 -7.48839169e-03  3.51473329e-02
  1.65962066e-01  7.7315226

### GENERATION OF EMBEDDING EXAMPLE
Here is an example of how an embedding is generated:

In [8]:
# get song data for a song
file_path = trainer.get_file_paths(input_dim=202, num_files=1)[0]
song = trainer.get_song_data_from_file(file_path)
nn_input = song.get_nn_input()

# feed forward through encoder
encoder.set_input(nn_input)
encoder.feed_forward()
output = encoder.get_output()

# print output
print("Embedding:")
print(output)

Embedding:
[  0.           0.          20.60261551   0.           0.
   0.           0.           0.           0.           0.
   0.           0.           0.           0.           0.
   0.           0.          39.94021703   0.           0.
   0.          29.7728619    0.           0.           0.
   0.           0.           0.           0.          32.44217515
   0.          84.34416309   0.           0.          37.43939632
   0.           0.           0.           0.           0.
   0.           0.           0.68151576   0.           0.
   0.          13.69149513   0.           0.         125.39734441
   0.           0.           0.           0.          41.01961594
   0.78961706   0.           0.           0.         106.10036715
   0.           0.           0.           0.        ]


### LATENT SPACE TRAVERSAL DEMO
Here is a small demo of the latent space traversal of the Recommender class. To keep visualization simple, a set of 2D points that are the corners of a hexagon are used. Traversal of the latent space embeddings generated by the encoder, such as the one above, are used to make recommendations. You will see in this demonstration an example of a recommendation path; the subsequent recommendation path if a recommendation is taken, and the regenerated path that is created if a recommendation is not taken.

In [9]:
curr_track = ('a', np.array([1, 1]))
tracks = [
    ('f', np.array([-1, 1])),
    ('c', np.array([1, -1])),
    ('b', np.array([2, 0])),
    ('e', np.array([-2, 0])),
    ('d', np.array([-1, -1])),
]

# test initializing recommendations
recommender = Recommender()
recommender.set_seed_track_info(curr_track)
recommender.init_selected_track_ids(tracks.copy())
recommender.init_recommendations()
recommendation_path_x = [i[1][0] for i in recommender._recommendations]
recommendation_path_y = [i[1][1] for i in recommender._recommendations]
plt.plot(recommendation_path_x, recommendation_path_y)
plt.scatter([1], [1])
plt.show()
print("Recommendations path after initialization:")
print(recommender._recommendations)
print()

# test getting next recommendation when previous recommendation taken
next_rec_id = recommender.get_next_recommendation_track_id(('b', np.array([2, 0])))
recommendation_path_x = [i[1][0] for i in recommender._recommendations]
recommendation_path_y = [i[1][1] for i in recommender._recommendations]
plt.plot(recommendation_path_x, recommendation_path_y)
plt.scatter([2], [0])
plt.show()
print("Next recommendation given user played b:")
print(next_rec_id)
print("Recommendation path:")
print(recommender._recommendations)
print()

Recommendations path after initialization:
[('b', array([2, 0])), ('c', array([ 1, -1])), ('d', array([-1, -1])), ('e', array([-2,  0])), ('f', array([-1,  1]))]

Next recommendation given user played b:
c
Recommendation path:
[('c', array([ 1, -1])), ('d', array([-1, -1])), ('e', array([-2,  0])), ('f', array([-1,  1]))]



### HELD-KARP OPTIMAL PATH GENERATION EXAMPLE
Below is an example of how the Held-Karp algorithm is used to generate the optimal path through a set of songs. What the "optimal path" is is decided by find the minimzation of the total Euclidean distance between the embeddings generated for the tracks. In order to do this, we will have to create a new recommender object. Normally, however, in the Algorhythm program, it is possible to change the traversal algorithm from greedy nearest neighbour to Held-Karp or vice versa by passing that in as the value to the get_next_recommendation() method. To demo this, we will use some songs from the test dataset.

In [10]:
# Instantiate recommender
recommender = Recommender(traversal_algorithm='optimal_path')

# Get list of song IDs
song_list_path = 'test_dataset/song_lists/precision_tests/a/10.txt'
song_ids = []
with open(song_list_path, 'r') as f:
    song_ids = [line.strip() for line in f.readlines()]

# Get paths to song data for songs with the given song IDs 
song_data_paths = [os.path.join('test_dataset/test_songs', song_id[2], song_id[3], song_id[4], f'{song_id}.h5') for song_id in song_ids]

# Get seed track data and give it to the recommender
seed_track_id = song_ids[0]
seed_song = trainer.get_song_data_from_file(song_data_paths[0])
seed_song_nn_input = seed_song.get_nn_input()
encoder.set_input(seed_song_nn_input)
encoder.feed_forward()
seed_song_embedding = encoder.get_output()
seed_track_info = (seed_track_id, seed_song_embedding)
recommender.set_seed_track_info(seed_track_info)

# Get data for the rest of the tracks and feed it into the recommender
selected_tracks_info = []
for song_path in song_data_paths[1:]:
    try:
        song = trainer.get_song_data_from_file(song_path)
    except:
        print(song_path)
        exit(0)
    song_nn_input = song.get_nn_input()
    encoder.set_input(song_nn_input)
    encoder.feed_forward()
    song_embedding = encoder.get_output()
    track_info = (os.path.splitext(os.path.basename(song_path))[0], song_embedding)
    selected_tracks_info.append(track_info)
recommender.init_selected_track_ids(selected_tracks_info=selected_tracks_info)

# Generate recommendations
recommender.init_recommendations()

# Get recommendations path
path = recommender.get_recommendation_path()
print("Seed track:")
print(path[0][0])
print("\nRecommendation path:")
for song in path[1:]:
    print(song[0])


Seed track:
SOCWPDW12B0B807859

Recommendation path:
SOFLJIY12A8C13FF69
SODJQUS12A6701D1D3
SOERCGN12A6D4F8B18
SOGKGVR12A6D4F803D
SOGKMRO12AB0180FE0
SOBDZKO12A67020664
SOFXDBI12A8AE48DA3
SOGYRGX12AB017B8BF
SOCUARC12A6701E94D


### TESTING DEMO
Below is a demo of how the pipeline is tested. A folder with a set of files, each containing song IDs where the song ID before a song shares at least one common listener according to the Echo Nest Taste Profile dataset, is used as the set of songs to test. Let us call these lists the "benchmark data". When testing precision, files for each of these songs is fetched, and in each of these files for the songs is a list of other songs that share at least one common listener according to the Echo Nest Taste Profile dataset, and if the song ID of the next song that is recommended is found in the file for the current song, it is taken as a positive, otherwise it is taken as a negative. When testing for Normalized Cumulative Gain, the files used instead contain a list of users that listened to the track, and the gain for a track recommended is taken as the Intersection Over Union (IoU) of the listeners listed in each of the song's files.

It is also possible to test the system using song lists that are generated randomly instead of the benchmark data simply by omitting the "--pregenerated" argument when running the file. However, for this demo we are going to use a set of pregenerated song lists.

In [ ]:
os.system('python3 test.py\
           --pregenerated \
           --test-precision \
           --test-dataset-path "/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/test_dataset/test_songs" \
           --song-plays-dataset-path "/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/test_dataset/songs_with_links_play_data" \
           --song-links-dataset-path "/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/test_dataset/song_links_dataset" \
           --pregenerated-lists-dataset-path "/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/test_dataset/song_lists/precision_tests/demo" \
           --output-path "/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/Statistics/Raw Data/Pregeneration/demo" \
           --nn-path "/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/networks/encoder_experiment_1/encoder.pkl" \
           --low-k 10 \
           --high-k 20 \
           --k-step 2 \
           --ncg-max-k 20 \
           --traversal-algorithm optimal_path')

Error: could not find neural network


0

# PIPELINE 2
Pipeline 2 makes use of the PyTorch library for the implementation of the autoencoder. The preprocessing and data preparation of this pipeline is exactly the same as that of Pipeline 1, so it will not be demonstrated here. The path generation is exactly the same as well (i.e. you can choose whether you would like to use the Greedy Nearest Neighbour algorithm or the Held-Karp algorithm to get the optimal path).

Note that the network inside the Autoencoder class contains both the encoder and decoder sections of the autoencoder. Below is a demonstration of how an autoencoder built in this pipeline is created:

In [ ]:
# Instantiate the network and print it
pt_nn = Autoencoder()
print(pt_nn)

### TRAINING DEMO
Below is a demo of the autoencoder created using PyTorch is trained. For training, a PyTorchTrainer object is used. When instantiating this object, you pass it the training hyperparameters like the learning rate, weight decay, momentum, and so forth. You can also select which optimizer you would like to use: Stochastic Gradient Descent (SGD) or Adam. The below example is using Adam as the optimizer.

In [ ]:
# Instantiate trainer object
pt_trainer = PyTorchTrainer(
    training_network=pt_nn,
    initial_lr=0.001,
    final_lr=0.00025,
    num_epochs=2,
    momentum=0.9,
    l2_regularization_lambda=0.001,
    dataset_path='./ProjectDataset',
    output_folder='pytorch_networks/demo',
)

# Perform training of the model for a single epoch
pt_trainer.train_model(opt_type='adam')

### LOADING PYTORCH MODEL FROM WEIGHTS FILE
Below is a demo of how an Autoencoder is loaded from a .pt file:

In [ ]:
# Instantiate model
pt_encoder = Autoencoder()
pt_encoder.load_state_dict(torch.load('pytorch_networks/experiment_3/best_val_loss_network.pt', weights_only=True))
print(pt_encoder)

### PYTORCH AUTOENCODER
Below is a demo of how the autoencoder from the second pipeline generates an embedding. To use the encoder, the encode() function is called, and this function just passes the input through the encoder section of the model.

In [ ]:
# Generate an embedding for a track
file_path = trainer.get_file_paths(input_dim=202, num_files=1)[0]
song = trainer.get_song_data_from_file(file_path)
x = torch.from_numpy(song.get_nn_input()).to(dtype=torch.float32)
output = pt_encoder.encode(x).detach().numpy()

# Print the output
print("Track data:")
print(song)
print("\nEmbedding for track: ", output)

### TESTING THE PYTORCH MODEL
Below is a demo of how the PyTorch model is tested. It is tested in almost exactly the same manner as the non-PyTorch pipeline's models. 

In [ ]:
os.system('python3 test_pytorch_model.py \
           --pregenerated \
           --test-precision \
           --test-dataset-path "/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/test_dataset/test_songs" \
           --song-plays-dataset-path "/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/test_dataset/songs_with_links_play_data" \
           --song-links-dataset-path "/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/test_dataset/song_links_dataset" \
           --pregenerated-lists-dataset-path "/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/test_dataset/song_lists/precision_tests/demo" \
           --output-path "/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/Statistics/Raw Data/Pregeneration/demo" \
           --nn-path "/home/troyxdp/Documents/University Work/HYP/HYP Source Code/Back End/pytorch_networks/experiment_3/best_val_loss_network.pt" \
           --low-k 10 \
           --high-k 20 \
           --k-step 2 \
           --ncg-max-k 20 \
           --traversal-algorithm optimal_path')